<a href="https://colab.research.google.com/github/Asuskf/from-nlp-to-agents/blob/embeddings/embeddings/Simulating_RAG_vector_mechanics/Simulating_RAG_vector_mechanics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab:Simulating RAG vector mechanics — A Conceptual RAG Simulation

# Why This Lab Matters

One of the biggest misconceptions about AI is that a model "decides to lie."

That is not how modern language models work.

Instead of searching a database for exact answers, an LLM represents words, ideas, and concepts as **vectors** inside a very large mathematical space called the **latent space**.

When a new question is close to regions containing many related concepts, the model usually produces coherent answers.

When a question lands in a sparse or ambiguous region of that space, the model still has to predict the next token—even though the available evidence is weaker.

This increases the probability of producing inaccurate or unsupported statements.

The purpose of this lab is **not** to eliminate hallucinations.

Instead, it helps us understand **why they can happen** and why techniques such as **Retrieval-Augmented Generation (RAG)** improve reliability by providing additional context.

---

# Learning Objectives

By the end of this lab you will understand:

- How an LLM can be viewed geometrically.
- Why similarity matters when generating text.
- How to measure a simple notion of confidence using cosine similarity.
- What a sparse region ("empty zone") represents.
- How Retrieval-Augmented Generation changes the position of a query in latent space.
- Why RAG adds knowledge without retraining the model.

---

# A Simplified View of the Latent Space

Real language models operate in spaces containing hundreds or thousands of dimensions.

For educational purposes, we will use only **three dimensions**.

| Dimension | Represents |
|------------|------------|
| X | Medicine |
| Y | Software |
| Z | Finance |

These dimensions are **not real embeddings**.

They simply help us visualize the mathematics.



# Level 0 — Measuring Geometric Confidence

We will define a simple confidence score using the cosine similarity between the query vector and every concept inside our knowledge base.

$$
\text{Confidence}
=
\max_{i\in KB}
\left(
\cos(\theta_{Q,i})
\right)
$$

where

- $$Q$$ is the query vector.
- $$KB$$ is the knowledge base.
- We keep only the highest similarity.

This is **not** the confidence score used internally by an LLM.

It is simply a geometric intuition that allows us to reason about proximity in latent space.

---

## Interpreting the Score

| Cosine Similarity | Interpretation |
|------------------|----------------|
| close to 1 | very similar |
| around 0.8 | reasonably close |
| below threshold | weak semantic proximity |

For this lab we define

In [4]:
CONFIDENCE_THRESHOLD = 0.75

This threshold is arbitrary and exists only for demonstration purposes.


# Level 1 — Building a Tiny Latent Universe

Let's create a miniature "memory" for our AI.


In [5]:
import numpy as np

def cosine_sim(v1, v2):
    return np.dot(v1, v2) / (
        np.linalg.norm(v1) *
        np.linalg.norm(v2)
    )

# 3D latent space:
# [Medicine, Software, Finance]

knowledge_base = {

    "clinical_diabetes":
        np.array([0.95,0.05,0.00]),

    "api_authentication":
        np.array([0.00,0.95,0.05]),

    "stock_dividends":
        np.array([0.00,0.10,0.90])

}

CONFIDENCE_THRESHOLD = 0.75

```text
                 Medicine

                    ● Diabetes



Software ● API                Finance ● Dividends
```

# Level 2 — Simulating an Ambiguous Query

Suppose a user asks

> "How does the clinical API impact dividend authentication?"

The question mixes concepts from multiple domains.

Our simplified embedding maps it to


In [6]:
query_vector = np.array([0.50,0.50,0.50])

``` text
Medicine

      ●


          Q



Software              Finance
```
The query is approximately in the middle of the space.

It is not clearly associated with any single knowledge cluster.


# Measuring the Nearest Concept


In [7]:
print("----- PHASE 1: UNGROUNDED QUERY -----")

best_match = None
max_confidence = 0

for concept, kb_vector in knowledge_base.items():

    similarity = cosine_sim(query_vector, kb_vector)

    if similarity > max_confidence:

        max_confidence = similarity
        best_match = concept

print(f"Nearest concept: {best_match}")

print(f"Geometric confidence: {max_confidence:.4f}")

if max_confidence < CONFIDENCE_THRESHOLD:

    print("\n⚠️ The query is far from any dense knowledge region.")

    print("This simplified simulation suggests that the model has weaker semantic support for generating a response.")

    print("The likelihood of hallucination is therefore higher.")

else:

    print("\nThe query is sufficiently close to an existing knowledge cluster.")

----- PHASE 1: UNGROUNDED QUERY -----
Nearest concept: stock_dividends
Geometric confidence: 0.6376

⚠️ The query is far from any dense knowledge region.
This simplified simulation suggests that the model has weaker semantic support for generating a response.
The likelihood of hallucination is therefore higher.


# What Does This Mean?

It is important to interpret the results correctly.

This simulator does **not** prove that the model will hallucinate.

Instead, it illustrates that

- the query has weak similarity to known concepts,
- semantic support is limited,
- generating a reliable answer becomes more difficult,
- the probability of unsupported statements increases.


# Level 3 — Semantic Grounding (The Core Idea Behind RAG)

Instead of immediately generating an answer, imagine we search a trusted document collection.

The retrieval system finds a document about API authentication.

We convert that document into another vector.

In [9]:
context_vector = np.array([0.00,0.98,0.02])

Instead of answering using only the original query, we combine both vectors.

---

# Vector Interpolation

The new query becomes

$$
\mathbf{Q}_{\text{anchored}}
=
(1-\alpha)\mathbf{Q}
+
\alpha\mathbf{C}
$$

where

- $$Q$$ is the original query.
- $$C$$ is the retrieved context.
- $$\alpha$$ controls how much influence the retrieved document has.

If

$$
\alpha=0
$$

the model ignores the retrieved document.

If

$$
\alpha=1
$$

the retrieved document completely dominates the query.

For this experiment we choose

In [10]:
alpha = 0.8

# Applying Semantic Grounding

In [11]:
print("\n----- PHASE 2: SEMANTIC GROUNDING -----")

context_vector = np.array([0.00,0.98,0.02])

alpha = 0.8

anchored_query = (

    (1-alpha)*query_vector

    +

    alpha*context_vector

)

print("Original query")

print(query_vector)

print("\nAnchored query")

print(anchored_query)

best_match = None

max_confidence = 0

for concept, kb_vector in knowledge_base.items():

    similarity = cosine_sim(

        anchored_query,

        kb_vector

    )

    if similarity > max_confidence:

        max_confidence = similarity

        best_match = concept

print(f"\nNearest concept: {best_match}")

print(f"New confidence: {max_confidence:.4f}")

if max_confidence < CONFIDENCE_THRESHOLD:

    print("\nThe query still remains in a sparse region.")

else:

    print("\n✅ The retrieved context moved the query closer to a dense knowledge cluster.")

    print("The model now has stronger semantic support for generating its answer.")


----- PHASE 2: SEMANTIC GROUNDING -----
Original query
[0.5 0.5 0.5]

Anchored query
[0.1   0.884 0.116]

Nearest concept: api_authentication
New confidence: 0.9908

✅ The retrieved context moved the query closer to a dense knowledge cluster.
The model now has stronger semantic support for generating its answer.


# What Changed?

Notice something important.

We never modified

- the model weights,
- the neural network,
- the knowledge base.

Instead, we changed only the **starting point** of the generation process.

Conceptually

```
Original Query

      Q
      ●

      │
      ▼

Retrieve Document

      │
      ▼

Anchored Query

      ● Q'
```

The query itself moved closer to relevant knowledge.

This is the central geometric intuition behind Retrieval-Augmented Generation.

---

# The RAG Pipeline

A simplified RAG architecture looks like this.

``` text
User Question

      │
      ▼

Embedding

      │
      ▼

Vector Database

      │
      ▼

Retrieve Documents

      │
      ▼

Append Context

      │
      ▼

LLM Generates Answer
```

Notice that the model is **not retrained**.

Instead, additional information is supplied during inference.